In [1]:
-- 18:
-- ================================================================
-- Author:      Nageen Saira
-- Procedure:   Retrieve initial row count
-- Create date: 04/12/2025
-- Description: Retrieves the initial row count from a specified table.
-- ================================================================
USE [G10_2];
GO

-- For example, if you have a test table named 'TestTable' in schema 'dbo'
SELECT COUNT(*) AS InitialRowCount
GO


Commands completed successfully.

(1 row affected)

Total execution time: 00:00:00.009

InitialRowCount
1


In [2]:
-- 19:
-- ================================================================
-- Author:      Nageen Saira
-- Procedure:   Retrieve foreign key relationships for a specific table
-- Create date: 04/12/2025
-- Description: Retrieves foreign key relationships for the 'DimOccupation' table.
-- ================================================================
USE [G10_2];
GO

SELECT
    fk.name                 AS FKName,
    OBJECT_NAME(fk.parent_object_id)   AS ReferencingTable,
    c1.name                AS ReferencingColumn,
    OBJECT_NAME(fk.referenced_object_id) AS ReferencedTable,
    c2.name                AS ReferencedColumn
FROM sys.foreign_keys fk
JOIN sys.foreign_key_columns fkc
    ON fk.object_id = fkc.constraint_object_id
JOIN sys.columns c1
    ON fkc.parent_object_id = c1.object_id
   AND fkc.parent_column_id = c1.column_id
JOIN sys.columns c2
    ON fkc.referenced_object_id = c2.object_id
   AND fkc.referenced_column_id = c2.column_id
WHERE OBJECT_NAME(fk.referenced_object_id) = 'DimOccupation';


Commands completed successfully.

(0 rows affected)

Total execution time: 00:00:00.093

FKName,ReferencingTable,ReferencingColumn,ReferencedTable,ReferencedColumn


In [3]:
-- 20:
-- ================================================================
-- Author:      Nageen Saira
-- Procedure:   Compare row counts using direct query and function
-- Create date: 04/12/2025
-- Description: Compares row count before an operation using direct query and custom function.
-- ================================================================
USE [G10_2];
GO

-- Direct COUNT(*)
SELECT COUNT(*) AS [DimOccupation_Before]
FROM [CH01-01-Dimension].[DimOccupation];

-- Using your GetRowCount function
SELECT dbo.GetRowCount('CH01-01-Dimension', 'DimOccupation') AS [FunctionCountBefore];


Commands completed successfully.

(1 row affected)

Total execution time: 00:00:00.088

DimOccupation_Before
0


FunctionCountBefore


: Msg 557, Level 16, State 2, Line 16
Only functions and some extended stored procedures can be executed from within a function.

In [5]:
-- 21:
-- ================================================================
-- Author:      Nageen Saira
-- Procedure:   Truncate DimOccupation table
-- Create date: 04/12/2025
-- Description: Truncates the 'DimOccupation' table in the specified schema.
-- ================================================================
EXEC [dbo].[sp_TruncateTableData]
     @SchemaName = N'CH01-01-Dimension',
     @TableName  = N'DimOccupation';
GO


TRUNCATE TABLE [CH01-01-Dimension].[DimOccupation];

Total execution time: 00:00:00.034

In [6]:
-- 22:
-- ================================================================
-- Author:      Nageen Saira
-- Procedure:   Compare row counts after truncation
-- Create date: 04/12/2025
-- Description: Compares row count after truncation using direct query and custom function.
-- ================================================================
-- Direct COUNT(*)
SELECT COUNT(*) AS [DimOccupation_After]
FROM [CH01-01-Dimension].[DimOccupation];

-- Using your GetRowCount function
SELECT dbo.GetRowCount('CH01-01-Dimension', 'DimOccupation') AS [FunctionCountAfter];


(1 row affected)

Total execution time: 00:00:00.039

DimOccupation_After
0


FunctionCountAfter


: Msg 557, Level 16, State 2, Line 13
Only functions and some extended stored procedures can be executed from within a function.

In [ ]:
-- 23:
-- ================================================================
-- Author:      Nageen Saira
-- Procedure:   Get row count with output parameter
-- Create date: 04/12/2025
-- Description: Retrieves the row count for a specified table and schema, returning the result via an output parameter.
-- ================================================================
CREATE OR ALTER PROCEDURE [dbo].[usp_GetRowCount]
(
    @SchemaName NVARCHAR(128),
    @TableName NVARCHAR(128),
    @RowCount INT OUTPUT
)
AS
BEGIN
    SET NOCOUNT ON;

    DECLARE @sql NVARCHAR(MAX);
    SET @sql = N'SELECT @cnt = COUNT(*) FROM ' 
               + QUOTENAME(@SchemaName) + N'.' + QUOTENAME(@TableName) + N';';

    EXEC sp_executesql @sql,
                       N'@cnt INT OUTPUT',
                       @cnt = @RowCount OUTPUT;
END;
GO


In [ ]:
-- 24:
-- ================================================================
-- Author:      Nageen Saira
-- Procedure:   Retrieve row count for DimOccupation table
-- Create date: 04/12/2025
-- Description: Executes the usp_GetRowCount procedure to get the row count for the DimOccupation table.
-- ================================================================
DECLARE @Count INT;
EXEC [dbo].[usp_GetRowCount]
     @SchemaName = 'CH01-01-Dimension',
     @TableName  = 'DimOccupation',
     @RowCount   = @Count OUTPUT;

SELECT @Count AS [DimOccupationRowCount];


In [7]:
-- 25:
-- ================================================================
-- Author:      Nageen Saira
-- Procedure:   Double-check row count for DimOccupation
-- Create date: 04/12/2025
-- Description: Executes a COUNT query to verify the row count in the DimOccupation table.
-- ================================================================
SELECT COUNT(*) AS [DoubleCheck]
FROM [CH01-01-Dimension].[DimOccupation];


(1 row affected)

Total execution time: 00:00:00.154

DoubleCheck
0


In [9]:
-- 26:
-- ================================================================
-- Author:      Nageen Saira
-- Procedure:   Retrieve row count for WorkflowSteps table
-- Create date: 04/12/2025
-- Description: Executes the usp_GetRowCount procedure to get the row count for the WorkflowSteps table.
-- ================================================================
DECLARE @MyCount INT;
EXEC [dbo].[usp_GetRowCount]
     @SchemaName = 'Process',
     @TableName  = 'WorkflowSteps',
     @RowCount   = @MyCount OUTPUT;

SELECT @MyCount AS [WorkflowStepsRowCount];


(1 row affected)

Total execution time: 00:00:00.013

WorkflowStepsRowCount
11


In [10]:
-- 27:
-- ================================================================
-- Author:      Nageen Saira
-- Procedure:   Retrieve row count for Data table
-- Create date: 04/12/2025
-- Description: Executes the usp_GetRowCount procedure to get the row count for the Data table in the CH01-01-Fact schema.
-- ================================================================
USE [G10_2];
GO

DECLARE @Count INT;

EXEC [dbo].[usp_GetRowCount]
     @SchemaName = N'CH01-01-Fact',
     @TableName  = N'Data',
     @RowCount   = @Count OUTPUT;

PRINT 'Row count for [CH01-01-Fact].[Data]: ' + CAST(@Count AS NVARCHAR(10));


Commands completed successfully.

Row count for [CH01-01-Fact].[Data]: 0

Total execution time: 00:00:00.024

In [11]:
-- 28:
-- ================================================================
-- Author:      Nageen Saira
-- Procedure:   Truncate Data table
-- Create date: 04/12/2025
-- Description: Executes the sp_TruncateTableData procedure to truncate the Data table in the CH01-01-Fact schema.
-- ================================================================
EXEC [dbo].[sp_TruncateTableData]
     @SchemaName = N'CH01-01-Fact',
     @TableName  = N'Data';


TRUNCATE TABLE [CH01-01-Fact].[Data];

Total execution time: 00:00:00.043

In [12]:
-- 29:
-- ================================================================
-- Author:      Nageen Saira
-- Procedure:   Truncate and check row count in Data table
-- Create date: 04/12/2025
-- Description: Executes the truncation procedure for the Data table in the CH01-01-Fact schema and prints the row count before and after the operation.
-- ================================================================
USE [G10_2];
GO

DECLARE @CountBefore INT, @CountAfter INT;

-- 1) Row count before any operation
EXEC [dbo].[usp_GetRowCount]
     @SchemaName = N'CH01-01-Fact',
     @TableName  = N'Data',
     @RowCount   = @CountBefore OUTPUT;

PRINT 'Data table row count BEFORE: ' + CAST(@CountBefore AS NVARCHAR(10));

-- 2) Call your truncation procedure (or a full load procedure if you have one)
EXEC [dbo].[sp_TruncateTableData]
     @SchemaName = N'CH01-01-Fact',
     @TableName  = N'Data';

-- Potentially call your load procedure here if you have something like usp_LoadData
-- EXEC [dbo].[usp_LoadData] @UserAuthKey = 1;

-- 3) Row count after the operation
EXEC [dbo].[usp_GetRowCount]
     @SchemaName = N'CH01-01-Fact',
     @TableName  = N'Data',
     @RowCount   = @CountAfter OUTPUT;

PRINT 'Data table row count AFTER: ' + CAST(@CountAfter AS NVARCHAR(10));


Commands completed successfully.

Data table row count BEFORE: 0

TRUNCATE TABLE [CH01-01-Fact].[Data];

Data table row count AFTER: 0

Total execution time: 00:00:00.024

In [ ]:
-- 30:
-- ================================================================
-- Author:      Nageen Saira
-- Procedure:   Load Data into [CH01-01-Fact].[Data] table
-- Create date: 04/12/2025
-- Description: This procedure logs the start of the data load, performs the data load from a source table into the Data table, and logs the completion of the operation.
-- ================================================================
CREATE OR ALTER PROCEDURE [dbo].[usp_LoadData]
    @UserAuthKey INT
AS
BEGIN
    SET NOCOUNT ON;

    -- 1) Log start of the data load
    EXEC [dbo].[usp_TrackWorkFlow] @UserAuthKey, N'Starting load for Data table';

    -- 2) Check row count, truncate (if necessary), and prepare for data load
    -- (Insert additional logic for truncation, validation, or pre-load checks as needed)
    
    -- Example: Check row count before truncation (optional)
    DECLARE @RowCountBefore INT;
    EXEC [dbo].[usp_GetRowCount]
         @SchemaName = N'CH01-01-Fact',
         @TableName  = N'Data',
         @RowCount   = @RowCountBefore OUTPUT;

    PRINT 'Row count before load: ' + CAST(@RowCountBefore AS NVARCHAR(10));

    -- Optional: Truncate the table if needed
    EXEC [dbo].[sp_TruncateTableData] @SchemaName = N'CH01-01-Fact', @TableName  = N'Data';

    -- 3) Insert data into [CH01-01-Fact].[Data] from source table
    INSERT INTO [CH01-01-Fact].[Data]
    (
        DataKey,  -- If this is an IDENTITY column, it should be omitted
        SomeMeasure,
        UserAuthorizationKey,
        DateAdded,
        DateOfLastUpdate
    )
    SELECT
        -- Assuming the DataKey is auto-generated (omit from SELECT)
        100,  -- example value or source data
        42,   -- example measure from source
        @UserAuthKey,
        GETDATE(),
        GETDATE()
    FROM [SomeSourceTable];  -- Replace with the actual source table name

    -- 4) Check row count after load, log completion
    DECLARE @RowCountAfter INT;
    EXEC [dbo].[usp_GetRowCount]
         @SchemaName = N'CH01-01-Fact',
         @TableName  = N'Data',
         @RowCount   = @RowCountAfter OUTPUT;

    PRINT 'Row count after load: ' + CAST(@RowCountAfter AS NVARCHAR(10));

    -- 5) Log completion of the data load
    EXEC [dbo].[usp_TrackWorkFlow] @UserAuthKey, N'Completed load for Data table';
END;
GO


In [13]:
--31:  ================================================================
-- Author:      Nageen Saira
-- Procedure:   Truncate data from CH01-01-Fact.Data table
-- Create date: 04/12/2025
-- Description: To truncate all records from the CH01-01-Fact.Data table 
--              using the stored procedure sp_TruncateTableData
-- ================================================================

USE [G10_2];
GO

-- 31:
EXEC [dbo].[sp_TruncateTableData] 
     @SchemaName = 'CH01-01-Fact', 
     @TableName  = 'Data';

Commands completed successfully.

TRUNCATE TABLE [CH01-01-Fact].[Data];

Total execution time: 00:00:00.026

In [ ]:
-- 32: ================================================================
-- Author:      Nageen Saira
-- Procedure:   Truncate and verify row count for CH01-01-Fact.Data
-- Create date: 04/12/2025
-- Description: This script checks the row count before and after 
--              truncating the CH01-01-Fact.Data table using stored 
--              procedures usp_GetRowCount and sp_TruncateTableData.
-- ================================================================

USE [G10_2];
GO

-- 32:
DECLARE @CountBefore INT, @CountAfter INT;

EXEC [dbo].[usp_GetRowCount]
     @SchemaName = 'CH01-01-Fact',
     @TableName  = 'Data',
     @RowCount   = @CountBefore OUTPUT;

PRINT 'Data row count BEFORE load: ' + CAST(@CountBefore AS NVARCHAR(10));

EXEC [dbo].[sp_TruncateTableData] 
     @SchemaName = 'CH01-01-Fact', 
     @TableName  = 'Data';

EXEC [dbo].[usp_GetRowCount]
     @SchemaName = 'CH01-01-Fact',
     @TableName  = 'Data',
     @RowCount   = @CountAfter OUTPUT;

PRINT 'Data row count AFTER truncate: ' + CAST(@CountAfter AS NVARCHAR(10));

In [14]:
-- 33: ================================================================
-- Author:      Nageen Saira
-- Procedure:   Full ETL Workflow - Logging, Truncate, Load, and Orchestration
-- Create date: 04/12/2025
-- Description: This script creates and tests stored procedures for 
--              logging ETL events, truncating and loading the Data table,
--              and orchestrating the ETL process end-to-end.
-- ================================================================

-- 33:
USE [G10_2];
GO

--------------------------------------------------------------------
-- 1. Create/Alter the workflow logging procedure
--------------------------------------------------------------------
CREATE OR ALTER PROCEDURE [dbo].[usp_TrackWorkFlow]
    @UserAuthKey INT,
    @WorkflowStepDescription NVARCHAR(255)
AS
BEGIN
    SET NOCOUNT ON;
    -- Insert a new workflow log entry
    INSERT INTO [Process].[WorkflowSteps]
        (UserAuthorizationKey, WorkflowStepDescription, StartingDateTime)
    VALUES
        (@UserAuthKey, @WorkflowStepDescription, GETDATE());
END;
GO

--------------------------------------------------------------------
-- 2. Create/Alter the Load Procedure for the Fact Table
--    This procedure logs the start, truncates the Data table,
--    loads new data (using a placeholder SELECT), and logs completion.
--------------------------------------------------------------------
CREATE OR ALTER PROCEDURE [dbo].[usp_LoadData]
    @UserAuthKey INT
AS
BEGIN
    SET NOCOUNT ON;

    -- Log the start of the Data table load
    EXEC [dbo].[usp_TrackWorkFlow] @UserAuthKey, N'Starting load for Data table';

    -- Truncate the target fact table
    EXEC [dbo].[sp_TruncateTableData] 
         @SchemaName = N'CH01-01-Fact', 
         @TableName  = N'Data';

    -- Insert new data into the Data table.
    INSERT INTO [CH01-01-Fact].[Data]
    (
        DataKey,           -- Remove if identity
        SomeMeasure,
        UserAuthorizationKey,
        DateAdded,
        DateOfLastUpdate
    )
    SELECT
        100,               -- Replace with dynamic data if needed
        42,
        @UserAuthKey,
        GETDATE(),
        GETDATE()
    FROM [SomeSourceTable]; -- Replace with your actual source

    -- Log the completion of the Data table load
    EXEC [dbo].[usp_TrackWorkFlow] @UserAuthKey, N'Completed load for Data table';
END;
GO

--------------------------------------------------------------------
-- 3. Create/Alter the Master ETL Orchestration Procedure
--------------------------------------------------------------------
CREATE OR ALTER PROCEDURE [dbo].[usp_RunETLProcess]
    @UserAuthKey INT
AS
BEGIN
    SET NOCOUNT ON;

    EXEC [dbo].[usp_TrackWorkFlow] @UserAuthKey, N'ETL process started';

    EXEC [dbo].[usp_LoadData] @UserAuthKey;

    -- Add other load procedures here if needed
    -- EXEC [dbo].[usp_LoadDimProductCategory] @UserAuthKey;
    -- EXEC [dbo].[usp_LoadDimProductSubcategory] @UserAuthKey;

    EXEC [dbo].[usp_TrackWorkFlow] @UserAuthKey, N'ETL process completed';
END;
GO

--------------------------------------------------------------------
-- 4. Test the Master ETL Procedure
--------------------------------------------------------------------
EXEC [dbo].[usp_RunETLProcess] @UserAuthKey = 1;
GO

-- Check workflow logs:
SELECT TOP (10) *
FROM [Process].[WorkflowSteps]
ORDER BY WorkFlowStepsKey DESC;
GO

-- Check inserted fact data:
SELECT TOP (10) *
FROM [CH01-01-Fact].[Data]
ORDER BY DataKey DESC;
GO

Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

TRUNCATE TABLE [CH01-01-Fact].[Data];

...

(10 rows affected)

(0 rows affected)

Total execution time: 00:00:00.133

WorkFlowStepDescription,WorkFlowStepTableRowCount,StartingDateTime,EndingDateTime,ClassTime,UserAuthorizationKey,WorkFlowStepsKey
ETL process completed,0,2025-04-14 00:33:43.1800000,2025-04-14 00:33:43.1700000,10:45,1,14
Starting load for Data table,0,2025-04-14 00:33:43.1700000,2025-04-14 00:33:43.1633333,10:45,1,13
ETL process started,0,2025-04-14 00:33:43.1666667,2025-04-14 00:33:43.1600000,10:45,1,12
Test via stored procedure,0,2025-04-14 00:05:18.9433333,2025-04-14 00:05:18.9500000,10:45,2,11
Test direct insert row,0,2025-04-14 00:05:18.9400000,NULL,10:45,1,10
ETL process completed,0,2025-04-13 22:36:39.0633333,2025-04-13 22:36:39.0666666,10:45,1,9
Starting load for Data table,0,2025-04-13 22:36:38.9833333,2025-04-13 22:36:38.9866666,10:45,1,8
ETL process started,0,2025-04-13 22:36:38.9800000,2025-04-13 22:36:38.9833333,10:45,1,7
ETL process completed,0,2025-04-13 04:29:43.5400000,2025-04-13 04:29:43.5266666,10:45,1,6
Starting load for Data table,0,2025-04-13 04:29:43.5333333,2025-04-13 04:29:43.5233333,10:45,1,5


DataKey,SomeMeasure,UserAuthorizationKey,DateAdded,DateOfLastUpdate
